# Volatility Drag: An Educational Walkthrough

When you compound random returns over time, the rate of growth a *typical* realised path achieves is *less* than the average return across the ensemble of all possible paths. The gap is **volatility drag**, and for Geometric Brownian Motion (GBM) it equals exactly $\tfrac{1}{2}\sigma^2$.

This notebook builds the concept from the ground up:

1. **Why drag exists at all** — Jensen's inequality and the Taylor expansion of $\log(1+r)$
2. **The closed-form GBM result** — Itô's lemma and the appearance of $\mu - \tfrac{1}{2}\sigma^2$
3. **How big drag is in practice** — the dimensionless ratio $\sigma^2 / (2\mu)$
4. **The regime map** — when drag is invisible, material, critical, or self-defeating
5. **How leverage interacts with drag** — quadratic scaling and the Kelly ceiling
6. **A worked example** — levering a high-Sharpe systematic strategy 5x
7. **Caveats** — when GBM is the wrong model

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from voldrag.analytics import (
    volatility_drag,
    geometric_mean_return,
    ensemble_mean,
    median_terminal,
    prob_path_below_mean,
)
from voldrag.gbm import simulate_gbm
from voldrag.experiments import coin_flip_game, sigma_sweep
from voldrag.plots import (
    plot_path_fan,
    plot_terminal_distribution,
    plot_sigma_sweep,
    plot_regime_map,
)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Why drag exists — the asymmetry of multiplication

A 50% loss requires a 100% gain to recover. That single fact contains the whole story.

If your wealth multiplies by $1+r$ and then by $1-r$, the result is $1 - r^2$ — not $1$. The arithmetic mean return is zero, but the typical realised result is a loss equal to $r^2$.

Generalising: $\log(1+r) \approx r - r^2/2$. The first term is what gets averaged into your drift; the **second term is volatility drag**. Nothing more mysterious than the second-order term of a Taylor series.

In [2]:
def two_period(r):
    return (1 + r) * (1 - r)

rows = []
for r in [0.05, 0.10, 0.20, 0.50]:
    rows.append({
        "r": r,
        "wealth after +r, -r": two_period(r),
        "loss": 1 - two_period(r),
        "r^2 (predicted)": r * r,
    })
pd.DataFrame(rows)

,r,"wealth after +r, -r",loss,r^2 (predicted)
0,0.05,0.9975,0.0025,0.0025
1,0.10,0.9900,0.0100,0.0100
2,0.20,0.9600,0.0400,0.0400
3,0.50,0.7500,0.2500,0.2500


The loss is *exactly* $r^2$. The bigger the swing, the bigger the gap between arithmetic break-even and your actual outcome. Variance is automatically a quadratic cost on compounding.